TASK-2 

Sentiment Analysis using NLP Pipeline & ML Models

In [14]:
pip install pandas numpy scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd

url = "https://raw.githubusercontent.com/dD2405/Twitter_Sentiment_Analysis/master/train.csv"
df = pd.read_csv(url)

df = df.rename(columns={"label": "sentiment", "tweet": "text"})
df['sentiment'] = df['sentiment'].map({0: "Negative", 1: "Positive"})

df = df.sample(5000, random_state=42)

print(df.head())
print(df['sentiment'].value_counts())

          id sentiment                                               text
12227  12228  Negative   @user âmy mom says my smile is captivatingâ...
14709  14710  Negative  in 3 days i will be meeting my sis-n-law, coun...
19319  19320  Negative  hating the conservative homophobes using this ...
4308    4309  Negative  awee if this doesn't  #scream   #friday #acewe...
24055  24056  Negative   fathersday  #fatherÃ¢ÂÂs #day #god! #ÃÂ« #...
sentiment
Negative    4663
Positive     337
Name: count, dtype: int64


In [16]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)

    tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

df['clean_text'] = df['text'].apply(preprocess_text)

df[['text','clean_text']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\varsh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,text,clean_text
12227,@user âmy mom says my smile is captivatingâ...,user mom says smile captivating says happy sun...
14709,"in 3 days i will be meeting my sis-n-law, coun...",days meeting sisnlaw couney bowers first level...
19319,hating the conservative homophobes using this ...,hating conservative homophobes using tragedy w...
4308,awee if this doesn't #scream #friday #acewe...,awee doesnt scream friday acewellstucker cynth...
24055,fathersday #fatherÃ¢ÂÂs #day #god! #ÃÂ« #...,fathersday fathers day god tony smith buy thin...


In [17]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()
X_bow = bow.fit_transform(df['clean_text'])

print("BoW Shape:", X_bow.shape)

BoW Shape: (5000, 11776)


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['clean_text'])

print("TF-IDF Shape:", X_tfidf.shape)

TF-IDF Shape: (5000, 11776)


In [19]:
from sklearn.model_selection import train_test_split

y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [20]:
#LogisticRegression

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [21]:
#Naive bayes

from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [22]:
#Decision tree

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_true, y_pred, model_name):
    print(f"\n{model_name}")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 3))
    print("Precision:", round(precision_score(y_true, y_pred, average='weighted'), 3))
    print("Recall:", round(recall_score(y_true, y_pred, average='weighted'), 3))
    print("F1 Score:", round(f1_score(y_true, y_pred, average='weighted'), 3))

evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")


Logistic Regression
Accuracy: 0.93
Precision: 0.935
Recall: 0.93
F1 Score: 0.898

Naive Bayes
Accuracy: 0.93
Precision: 0.935
Recall: 0.93
F1 Score: 0.898

Decision Tree
Accuracy: 0.937
Precision: 0.925
Recall: 0.937
F1 Score: 0.926


Comparison & Insights
Explain best preprocessing steps, best vectorization, best model, and trade-offs.


1.Logistic Regression performed best because it handles text features well.

2.Naive Bayes is fast but assumes independence between words.

3.Decision Tree tends to overfit on text data.

4.TF-IDF gave better results than Bag of Words because it considers word importance.